# 03 — Learned Retrieval Baselines: Item2Vec + BPR

This notebook moves from heuristic co-visitation to learned item representations.

It covers:
- Item2Vec session embeddings
- BPR matrix factorization for implicit feedback
- shared Recall@K / MRR@K evaluation
- model artifact saving

Goal: determine whether learned retrieval improves candidate generation before moving to Two-Tower retrieval.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

from gensim.models import Word2Vec
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load processed data

The notebook reuses the Parquet files produced by Notebook 02 so we do not parse the large JSONL file again.


In [ ]:
event_files = sorted(PROCESSED_DIR.glob("otto_train_*_sessions.parquet"))
valid_files = sorted(PROCESSED_DIR.glob("otto_validation_*_sessions.parquet"))

if not event_files or not valid_files:
    raise FileNotFoundError("Run Notebook 02 first.")

events = pd.read_parquet(event_files[-1])
valid_examples = pd.read_parquet(valid_files[-1])

print(events.shape)
print(valid_examples.shape)


## 3. Recreate temporal training split

In [ ]:
VALID_FRACTION = 0.10

session_meta = (
    events.groupby("session")
    .agg(end_ts=("ts", "max"))
    .reset_index()
    .sort_values("end_ts")
    .reset_index(drop=True)
)

split_idx = int(len(session_meta) * (1 - VALID_FRACTION))
train_session_ids = set(session_meta.iloc[:split_idx]["session"])

train_events = events[
    events["session"].isin(train_session_ids)
].copy()

print(f"Training sessions: {train_events['session'].nunique():,}")
print(f"Training events:   {len(train_events):,}")
print(f"Training items:    {train_events['aid'].nunique():,}")


## 4. Shared ranking metrics

In [ ]:
def evaluate_model(examples, recommend_fn, k=20, max_examples=None):
    subset = examples if max_examples is None else examples.head(max_examples)

    hits = []
    reciprocal_ranks = []

    for row in subset.itertuples(index=False):
        recommendations = recommend_fn(row, k=k)

        if row.target_aid in recommendations[:k]:
            rank = recommendations[:k].index(row.target_aid) + 1
            hits.append(1.0)
            reciprocal_ranks.append(1.0 / rank)
        else:
            hits.append(0.0)
            reciprocal_ranks.append(0.0)

    return {
        f"Recall@{k}": float(np.mean(hits)),
        f"HitRate@{k}": float(np.mean(hits)),
        f"MRR@{k}": float(np.mean(reciprocal_ranks)),
    }


# Part A — Item2Vec

Treat each session as a sentence and each item ID as a token. Items appearing in similar session contexts receive similar embeddings.


## 5. Build session sequences

In [ ]:
train_sorted = train_events.sort_values(["session", "ts", "position"])

MAX_SEQUENCE_LENGTH = 100

session_sequences = (
    train_sorted.groupby("session")["aid"]
    .apply(lambda x: [str(int(v)) for v in x][-MAX_SEQUENCE_LENGTH:])
    .tolist()
)

session_sequences = [
    seq for seq in session_sequences if len(seq) >= 2
]

print(f"Sequences: {len(session_sequences):,}")
print(session_sequences[0][:20])


## 6. Train Item2Vec

In [ ]:
item2vec = Word2Vec(
    sentences=session_sequences,
    vector_size=64,
    window=10,
    min_count=3,
    sg=1,          # Skip-Gram
    negative=10,   # sampled negatives
    workers=4,
    epochs=5,
    seed=RANDOM_SEED,
)

print(f"Vocabulary size: {len(item2vec.wv):,}")


## 7. Build a recency-weighted session vector

Recent items receive more weight because they better represent current intent.


In [ ]:
global_popularity = (
    train_events["aid"]
    .value_counts()
    .index.astype(int)
    .tolist()
)

def item2vec_session_vector(observed_aids):
    vectors = []
    weights = []

    for recency_rank, aid in enumerate(reversed(observed_aids)):
        key = str(int(aid))

        if key not in item2vec.wv:
            continue

        vectors.append(item2vec.wv[key])
        weights.append(1.0 / (recency_rank + 1))

    if not vectors:
        return None

    return np.average(
        np.vstack(vectors),
        axis=0,
        weights=np.asarray(weights),
    )


## 8. Item2Vec recommender

In [ ]:
def recommend_item2vec(row, k=20):
    session_vector = item2vec_session_vector(row.observed_aids)

    if session_vector is None:
        return global_popularity[:k]

    candidates = item2vec.wv.similar_by_vector(
        session_vector,
        topn=k * 5,
    )

    recommendations = []

    for item_key, similarity in candidates:
        aid = int(item_key)

        if aid not in recommendations:
            recommendations.append(aid)

        if len(recommendations) >= k:
            break

    return recommendations[:k]


## 9. Evaluate Item2Vec

In [ ]:
item2vec_metrics = evaluate_model(
    valid_examples,
    recommend_item2vec,
    k=20,
)

pd.Series(item2vec_metrics, name="Item2Vec")


# Part B — BPR

BPR learns pairwise preferences from implicit feedback: interacted items should rank above sampled non-interacted items.


## 10. Build sparse interaction matrix

In [ ]:
EVENT_WEIGHTS = {
    "clicks": 1.0,
    "carts": 3.0,
    "orders": 6.0,
}

session_ids = train_events["session"].unique()
item_ids = train_events["aid"].unique()

session_to_idx = {sid: i for i, sid in enumerate(session_ids)}
item_to_idx = {aid: i for i, aid in enumerate(item_ids)}
idx_to_item = np.asarray(item_ids)

tmp = train_events[["session", "aid", "type"]].copy()
tmp["weight"] = tmp["type"].map(EVENT_WEIGHTS).astype(np.float32)

agg = (
    tmp.groupby(["session", "aid"], as_index=False)["weight"]
    .sum()
)

rows = agg["session"].map(session_to_idx).to_numpy()
cols = agg["aid"].map(item_to_idx).to_numpy()
data = agg["weight"].to_numpy(dtype=np.float32)

interaction_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(session_to_idx), len(item_to_idx)),
)

print(interaction_matrix.shape)
print(f"Non-zero interactions: {interaction_matrix.nnz:,}")


## 11. Train BPR

In [ ]:
bpr = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.05,
    regularization=0.01,
    iterations=50,
    random_state=RANDOM_SEED,
)

bpr.fit(interaction_matrix)


## 12. Build validation-session vectors from learned item factors

Validation sessions are unseen during training, so we approximate their representation using a recency-weighted average of item factors.


In [ ]:
item_factors = bpr.item_factors

factor_norms = np.linalg.norm(
    item_factors,
    axis=1,
    keepdims=True,
)

normalized_item_factors = (
    item_factors
    / np.clip(factor_norms, 1e-12, None)
)

def bpr_session_vector(observed_aids):
    vectors = []
    weights = []

    for recency_rank, aid in enumerate(reversed(observed_aids)):
        idx = item_to_idx.get(aid)

        if idx is None:
            continue

        vectors.append(item_factors[idx])
        weights.append(1.0 / (recency_rank + 1))

    if not vectors:
        return None

    return np.average(
        np.vstack(vectors),
        axis=0,
        weights=np.asarray(weights),
    )


## 13. BPR recommender

In [ ]:
def recommend_bpr(row, k=20):
    session_vector = bpr_session_vector(row.observed_aids)

    if session_vector is None:
        return global_popularity[:k]

    session_vector = (
        session_vector
        / max(np.linalg.norm(session_vector), 1e-12)
    )

    scores = normalized_item_factors @ session_vector

    n_candidates = min(k * 5, len(scores))

    candidate_idx = np.argpartition(
        -scores,
        n_candidates - 1,
    )[:n_candidates]

    candidate_idx = candidate_idx[
        np.argsort(-scores[candidate_idx])
    ]

    return [
        int(idx_to_item[i])
        for i in candidate_idx[:k]
    ]


## 14. Evaluate BPR

In [ ]:
bpr_metrics = evaluate_model(
    valid_examples,
    recommend_bpr,
    k=20,
)

pd.Series(bpr_metrics, name="BPR")


## 15. Compare learned retrieval baselines

In [ ]:
retrieval_results = pd.DataFrame([
    {"model": "item2vec", **item2vec_metrics},
    {"model": "bpr", **bpr_metrics},
]).set_index("model")

retrieval_results


## 16. Save artifacts

In [ ]:
item2vec.save(
    str(MODEL_DIR / "item2vec.model")
)

with open(MODEL_DIR / "bpr_model.pkl", "wb") as f:
    pickle.dump(
        {
            "model": bpr,
            "item_to_idx": item_to_idx,
            "idx_to_item": idx_to_item,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

retrieval_results.to_csv(
    PROCESSED_DIR / "learned_retrieval_results.csv"
)

print("Saved.")


## Next

Notebook 04 introduces a neural Two-Tower retriever and FAISS ANN search. The key metric becomes candidate Recall@100/200 rather than only Top-20 ordering.
